<a href="https://colab.research.google.com/github/sonashah02/retail-churn-market-basket-analysis/blob/main/ChurnModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
cleaned_data = pd.read_csv('/content/cleaned_orders1.csv')

In [ ]:
cleaned_data.head()

In [ ]:
# Rename Name to Order ID
cleaned_data = cleaned_data.rename(columns={'Name': 'OrderID'})

In [ ]:
# Forward fill customer names and order total
cleaned_data['Billing Name'] = cleaned_data.groupby('OrderID')['Billing Name'].transform('first')
cleaned_data['Total'] = cleaned_data.groupby('OrderID')['Total'].transform('first')

In [ ]:
# Make dates datetimes using a function

def make_datetime(df, column):
    df[column] = pd.to_datetime(df[column])
    return df

cleaned_data = make_datetime(cleaned_data, 'Created at')
cleaned_data = make_datetime(cleaned_data,'Fulfilled at')
cleaned_data = make_datetime(cleaned_data,'Cancelled at')
cleaned_data = make_datetime(cleaned_data,'Paid at')

In [ ]:
# Rename columns
cleaned_data = cleaned_data.rename(columns={'Created at': 'order_date'})
cleaned_data = cleaned_data.rename(columns={'Billing Name': 'billing_name'})

In [ ]:
# Remove the # before every OrderID
cleaned_data.loc[:, 'OrderID'] = cleaned_data['OrderID'].str.replace('#', '')
display(cleaned_data.head())

In [ ]:
# Remove NEW/RETIRED/PREORDER from Lineitem name

cleaned_data.loc[:, 'Lineitem name'] = cleaned_data['Lineitem name'].str.replace('NEW - ', '')
cleaned_data.loc[:, 'Lineitem name'] = cleaned_data['Lineitem name'].str.replace('RETIRED - ', '')
cleaned_data.loc[:, 'Lineitem name'] = cleaned_data['Lineitem name'].str.replace('PREORDER - ', '')

In [ ]:
# Only keep lines of orders created 2024-2026

cleaned_data_2024 = cleaned_data[cleaned_data['order_date'].dt.year >= 2024]

## Customer Churn Prediction

**Objective:** Predict which customers are at risk of not returning within 6 months,
using transaction history from a small retail business's POS and web order data.

**Approach:** Built customer-level features (recency, frequency, monetary value,
category diversity, order interval variability, order value trend) from a Jan 2024–Dec 2025
feature window, and defined churn as no purchase in the following 6-month outcome window
(Jan–Jun 2026). Compared Logistic Regression and Random Forest classifiers.

### Churn Model

In [ ]:
cutoff_date = pd.Timestamp('2026-01-01')

feature_df = cleaned_data_2024[cleaned_data_2024['order_date'] < cutoff_date]
outcome_df = cleaned_data_2024[cleaned_data_2024['order_date'] >= cutoff_date]

print(f"Feature window: {feature_df['order_date'].min()} to {feature_df['order_date'].max()}")
print(f"Outcome window: {outcome_df['order_date'].min()} to {outcome_df['order_date'].max()}")
print(f"Customers in feature window: {feature_df['billing_name'].nunique()}")

Feature window: 2024-01-04 03:14:59 to 2025-12-31 17:07:51
Outcome window: 2026-01-02 11:16:13 to 2026-06-11 17:48:53
Customers in feature window: 1187


In [ ]:
# Step 1: Build order_level from feature_df, excluding $0 promotional orders for items owners gave for free
order_level = feature_df[feature_df['Total'] > 0].drop_duplicates(subset=['OrderID'])[
    ['billing_name', 'OrderID', 'order_date', 'Total']
]

# Check
print(feature_df.shape[0], order_level.shape[0], order_level['OrderID'].duplicated().sum())

5477 1553 0


In [ ]:
# Step 2: Build customer_features from order_level
customer_features = order_level.groupby('billing_name').agg(
    first_order=('order_date', 'min'),
    last_order=('order_date', 'max'),
    num_orders=('OrderID', 'nunique'),
    total_spend=('Total', 'sum'),
    avg_order_value=('Total', 'mean')
).reset_index()

#Recency days = time since last order and Jan 2026, tenure days = time since first order and Jan 2026
customer_features['recency_days'] = (cutoff_date - customer_features['last_order']).dt.days
customer_features['tenure_days'] = (cutoff_date - customer_features['first_order']).dt.days

#Vendor diversity
cat_diversity = feature_df.groupby('billing_name')['Vendor'].nunique().reset_index()
cat_diversity.columns = ['billing_name', 'category_diversity']
customer_features = customer_features.merge(cat_diversity, on='billing_name')

In [ ]:
# Step 3: Purchase interval variability (from deduped order_level)
order_dates_per_customer = order_level.groupby('billing_name')['order_date'].apply(
    lambda x: x.sort_values().diff().dt.days.dropna()
)

interval_std = order_dates_per_customer.groupby('billing_name').std().reset_index()
interval_std.columns = ['billing_name', 'interval_std_days']

customer_features = customer_features.merge(interval_std, on='billing_name', how='left')
# if no interval_std, fill with 0
customer_features['interval_std_days'] = customer_features['interval_std_days'].fillna(0)
# Create column for single order
customer_features['is_single_order'] = (customer_features['num_orders'] == 1).astype(int)

In [ ]:
# Step 4: Churn target
returned_customers = set(outcome_df['billing_name'].unique())
customer_features['churned'] = (~customer_features['billing_name'].isin(returned_customers)).astype(int)

print(customer_features.shape)
print(customer_features['churned'].value_counts(normalize=True))
print(customer_features['churned'].value_counts())
customer_features.describe()

(1176, 12)
churned
1    0.817177
0    0.182823
Name: proportion, dtype: float64
churned
1    961
0    215
Name: count, dtype: int64


,first_order,last_order,num_orders,total_spend,avg_order_value,recency_days,tenure_days,category_diversity,interval_std_days,is_single_order,churned
count,1176,1176,1176.000000,1176.000000,1176.000000,1176.000000,1176.000000,1176.000000,1176.000000,1176.000000,1176.000000
mean,2025-10-13 04:21:54.962585088,2025-10-20 16:24:57.253401600,1.229592,115.182874,91.982835,71.901361,79.401361,2.367347,1.278325,0.844388,0.817177
min,2024-01-04 03:14:59,2024-01-04 03:14:59,1.000000,4.270000,4.270000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2025-11-14 15:08:46.500000,2025-11-18 13:23:30.500000,1.000000,37.750000,35.530625,12.000000,13.000000,1.000000,0.000000,1.000000,1.000000
50%,2025-12-06 11:21:59.500000,2025-12-08 14:58:23,1.000000,71.280000,64.640000,23.000000,25.000000,2.000000,0.000000,1.000000,1.000000
75%,2025-12-18 13:11:46.750000128,2025-12-19 13:13:23.750000128,1.000000,133.400000,111.455000,43.000000,47.000000,3.000000,0.000000,1.000000,1.000000
max,2025-12-31 17:07:51,2025-12-31 17:07:51,17.000000,3134.240000,2006.890000,727.000000,727.000000,19.000000,385.373196,1.000000,1.000000
std,NaN,NaN,0.821006,172.322181,106.813473,141.140879,151.815326,1.885512,17.638330,0.362642,0.386686


In [ ]:
# measures if a customer's orders are growing or shrinking over time

from scipy import stats

def calc_trend(group):
    if len(group) < 2:
        return 0  # can't compute a trend with fewer than 2 orders
    x = range(len(group))
    y = group.sort_values('order_date')['Total'].values
    slope, _, _, _, _ = stats.linregress(x, y)
    return slope

order_trend = order_level.groupby('billing_name').apply(calc_trend).reset_index()
order_trend.columns = ['billing_name', 'order_value_trend']

customer_features = customer_features.merge(order_trend, on='billing_name', how='left')

/tmp/ipykernel_794/982025907.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  order_trend = order_level.groupby('billing_name').apply(calc_trend).reset_index()


In [ ]:
customer_features['order_value_trend'].describe()

,order_value_trend
count,1176.000000
mean,-5.501520
std,37.233437
min,-388.530000
25%,0.000000
50%,0.000000
75%,0.000000
max,260.270000


## Data Quality Notes

Several data quality issues were identified and corrected during feature engineering:
- Transaction data was stored at the line-item level; order-level aggregations required
  deduplication by OrderID to avoid inflating totals and order counts.
- ~241 zero-dollar "orders" were identified as free promotional items logged as separate
  transactions and excluded from order counts and interval calculations.
- The business migrated in-store POS systems in Jan 2026, meaning in-store transactions
  prior to this date are undercounted in this dataset. This is a known limitation: churn
  estimates may be less reliable for customers whose primary channel was in-store prior
  to the migration.

In [ ]:
#confirm that train and test churn rates are similar

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

feature_cols = ['recency_days', 'tenure_days', 'num_orders', 'total_spend',
                 'avg_order_value', 'category_diversity', 'interval_std_days',
                 'is_single_order', 'order_value_trend']

X = customer_features[feature_cols]
y = customer_features['churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Train churn rate: {y_train.mean():.3f}, Test churn rate: {y_test.mean():.3f}")

Train: 940, Test: 236
Train churn rate: 0.817, Test churn rate: 0.818


Logistic Regression

In [ ]:
from sklearn.preprocessing import StandardScaler

# Logistic regression benefits from scaled features (recency days could be in the hundreds but is single order is just 0/1. these differences in column values make scaling important)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000) # balanced ensures pay more attention to the minority class
log_reg.fit(X_train_scaled, y_train)

y_pred = log_reg.predict(X_test_scaled)
y_pred_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.3f}")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.35      0.65      0.45        43
           1       0.90      0.73      0.80       193

    accuracy                           0.71       236
   macro avg       0.62      0.69      0.63       236
weighted avg       0.80      0.71      0.74       236

ROC-AUC: 0.746
[[ 28  15]
 [ 53 140]]


Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    max_depth=6  # keeps trees from overfitting given the small dataset
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_pred_proba_rf = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_rf):.3f}")
print(confusion_matrix(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.34      0.53      0.41        43
           1       0.88      0.77      0.82       193

    accuracy                           0.72       236
   macro avg       0.61      0.65      0.62       236
weighted avg       0.78      0.72      0.75       236

ROC-AUC: 0.690
[[ 23  20]
 [ 45 148]]


Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print(importance_df)

              feature  importance
0        recency_days    0.229359
3         total_spend    0.222720
4     avg_order_value    0.181220
1         tenure_days    0.179237
5  category_diversity    0.072029
8   order_value_trend    0.065142
2          num_orders    0.026268
7     is_single_order    0.015165
6   interval_std_days    0.008861


## Results

| Model | Precision (retained) | Recall (retained) | ROC-AUC |
|---|---|---|---|
| Logistic Regression | 0.35 | 0.65 | 0.746 |
| Random Forest | 0.34 | 0.53 | 0.690 |

Logistic Regression outperformed Random Forest on both recall and AUC, suggesting the
relationship between customer behavior and churn risk in this dataset is largely linear —
Random Forest's capacity to model nonlinear interactions did not improve results, and may
have overfit given the limited training size (940 customers).

**Top predictive features:** recency (days since last order), total spend, average order
value, and tenure were the strongest predictors of churn. Engineered features like purchase
interval variability contributed minimally, likely because 84% of customers in this dataset
placed only a single order in the feature window — limiting the value of repeat-purchase
behavioral signals.

**Limitations:** With only 215 non-churned examples in the full dataset, model evaluation
has meaningful variance. Given the high proportion of one-time purchasers, this model is
better understood as predicting "likelihood of any return purchase" rather than churn from
an established repeat-purchase relationship.